In [ ]:
!pip -q install -U transformers datasets accelerate evaluate rouge_score sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
!pip install -U "peft>=0.17.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.21.1
    Uninstalling peft-0.21.1:
      Successfully uninstalled peft-0.21.1


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import torch

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# **1- Model and dataset**

In [ ]:
MODEL_NAME = "facebook/bart-base"
DATASET_NAME = "EdinburghNLP/xsum"

#use subset if training takes long
USE_SUBSET = True
TRAIN_SUBSET_SIZE = 100000
EVAL_SUBSET_SIZE = 1000

#OUTPUT_DIR = "/content/xsum_bart_lora"

In [ ]:
from datasets import load_dataset

#loading dataset
dataset = load_dataset(DATASET_NAME)

print("\nDataset splits:")
for split_name, split in dataset.items():
    print(f"Original {split_name}: {len(split):,} examples")

train_dataset = dataset["train"].shuffle(seed=SEED)

if USE_SUBSET:
    train_dataset = train_dataset.select(
        range(min(TRAIN_SUBSET_SIZE, len(train_dataset)))
    )

eval_dataset = dataset["validation"].shuffle(seed=SEED)

if EVAL_SUBSET_SIZE is not None:
    eval_dataset = eval_dataset.select(
        range(min(EVAL_SUBSET_SIZE, len(eval_dataset)))
    )
print("using sample length as below for training:")
print("Training:", len(train_dataset))
print("Validation:", len(eval_dataset))

README.md:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  300MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]


Dataset splits:
Original train: 204,045 examples
Original validation: 11,332 examples
Original test: 11,334 examples
Training: 100000
Validation: 1000
Test: 11334


# **2- Tokenization**

In [ ]:
#calculating some statistics on tokens
from transformers import AutoTokenizer, DataCollatorForSeq2Seq
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LENGTH_SAMPLE_SIZE = 10000

# Sample from the original training split
length_sample = dataset["train"].shuffle(
    seed=SEED
).select(
    range(min(LENGTH_SAMPLE_SIZE, len(dataset["train"])))
)

# Count tokens without truncation
def count_token_lengths(batch):
    source = tokenizer(
        batch["document"],
        truncation=False,
        add_special_tokens=True,
    )

    target = tokenizer(
        text_target=batch["summary"],
        truncation=False,
        add_special_tokens=True,
    )

    return {
        "source_length": [len(ids) for ids in source["input_ids"]],
        "target_length": [len(ids) for ids in target["input_ids"]],
    }

length_sample = length_sample.map(
    count_token_lengths,
    batched=True,
    desc="Measuring token lengths",
)

# Convert lengths to numpy arrays
source_lengths = np.array(length_sample["source_length"])
target_lengths = np.array(length_sample["target_length"])

# Calculate descriptive statistics
def length_statistics(lengths):
    quantiles = np.quantile(
        lengths, [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )

    return {
        "count": len(lengths),
        "mean": np.mean(lengths),
        "std": np.std(lengths),
        "min": np.min(lengths),
        "Q1 (25%)": quantiles[0],
        "median (50%)": quantiles[1],
        "Q3 (75%)": quantiles[2],
        "P90": quantiles[3],
        "P95": quantiles[4],
        "P99": quantiles[5],
        "max": np.max(lengths),
    }

stats_df = pd.DataFrame({
    "Source article": length_statistics(source_lengths),
    "Target summary": length_statistics(target_lengths),
}).T

display(stats_df.round(2))

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Measuring token lengths:   0%|          | 0/10000 [00:00<?, ? examples/s]

,count,mean,std,min,Q1 (25%),median (50%),Q3 (75%),P90,P95,P99,max
Source article,10000.0,484.46,381.47,2.0,230.0,386.0,629.0,973.0,1206.05,1743.03,8387.0
Target summary,10000.0,28.18,6.69,3.0,24.0,28.0,32.0,36.0,39.00,48.00,101.0


In [ ]:
# 1024 covers P90 of XSum articles. 128 is well above the token summary length.
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    inputs = tokenizer(
        examples["document"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    targets = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

# batched=True is faster
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train",
)

tokenized_eval = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=eval_dataset.column_names,
    desc="Tokenizing validation",
)

print("Train examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_eval))

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

Tokenizing train:   0%|          | 0/100000 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train examples: 100000
Validation examples: 1000


# **3- Configure LoRA**

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model

print("Loading BART-base")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Remove BART's default max_length=20
model.generation_config.max_length = None

model.config.use_cache = False
#model.gradient_checkpointing_enable()
#model.enable_input_require_grads()

#train small adapters on attention projections only
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


Loading BART-base


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]


Num trainable parameter:
trainable params: 663,552 || all params: 140,083,968 || trainable%: 0.4737


# **4- Evaluation metrics**

In [ ]:
import evaluate

#Compute ROUGE on generated summaries
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Replace -100 padding with pad id so decode works
    predictions = np.where(predictions == -100, tokenizer.pad_token_id, predictions)
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_predictions = [x.strip() for x in decoded_predictions]
    decoded_labels = [x.strip() for x in decoded_labels]

    scores = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True,
    )

    return {key: round(value * 100, 4) for key, value in scores.items()}

print("ROUGE-1, ROUGE-2, and ROUGE-L metrics ready.")


ROUGE-1, ROUGE-2, and ROUGE-L metrics ready.


# **5- Training callback**

In [ ]:
from transformers import TrainerCallback

#prints elapsed time and GPU memory and other information at the start of training, on every log step, and at the end
class TrainingCallback(TrainerCallback):

    def on_train_begin(self, args, state, control, **kwargs):
        # Record the start time so we can report elapsed minutes later
        self.start_time = time.time()
        print(f"\nTraining started | Epochs: {args.num_train_epochs}")
        self.print_gpu()

    def print_gpu(self):
        #Print current and peak GPU memory usage
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            peak = torch.cuda.max_memory_allocated() / 1024**3
            total = torch.cuda.get_device_properties(0).total_memory / 1024**3

            print(
                f"GPU memory: {allocated:.2f}/{total:.2f} GB used | "
                f"Reserved: {reserved:.2f} GB | Peak: {peak:.2f} GB"
            )
        else:
            print("GPU not available")

    def on_log(self, args, state, control, logs=None, **kwargs):
        #Called by the Trainer at each logging step.
        if logs:
            progress = 100 * state.global_step / state.max_steps
            print(f"\nStep {state.global_step}/{state.max_steps} ({progress:.1f}%)")

            for key, value in logs.items():
                if isinstance(value, (int, float)):
                    print(f"{key}: {value:.4f}")

            elapsed = (time.time() - self.start_time) / 60
            print(f"Elapsed: {elapsed:.1f} min")
            self.print_gpu()
            torch.cuda.reset_peak_memory_stats()

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = (time.time() - self.start_time) / 60
        print(f"\nTraining finished | Total time: {elapsed:.1f} min")
        self.print_gpu()

# **6- Trainer setup**

In [ ]:
# hyperparameters
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2    # effective batch size 32

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100

LOGGING_STEPS = 500
EVAL_STEPS = 750   # 4 evals per epoch on 100k examples
SAVE_STEPS = 750

GENERATION_MAX_LENGTH = 128
GENERATION_NUM_BEAMS = 2

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# TrainingArguments: batch/LR schedule, eval, and generation settings
training_args = Seq2SeqTrainingArguments(

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",

    fp16=True,
    #gradient_checkpointing=True,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_LENGTH,
    generation_num_beams=GENERATION_NUM_BEAMS,

    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,

    report_to="none",
    seed=SEED,
    data_seed=SEED,

    dataloader_num_workers=0,
    eval_accumulation_steps=8,
)

# adding model, tokenized data, collator, ROUGE metric, and logging callback together
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[TrainingCallback()],
)

print("Trainer initialized.")
print(f"Training examples: {len(tokenized_train):,}")
print(f"Validation examples: {len(tokenized_eval):,}")

Trainer initialized.
Training examples: 100,000
Validation examples: 1,000
trainable params: 663,552 || all params: 140,083,968 || trainable%: 0.4737


# **7- Run training**

In [ ]:
print("Starting LoRA fine-tuning")

train_result = trainer.train()

print("\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

Starting LoRA fine-tuning...

Training started | Epochs: 1
GPU memory: 0.52/14.56 GB used | Reserved: 0.58 GB | Peak: 0.52 GB


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
750,5.226796,2.072565,34.176800,12.345600,27.208700,27.245500
1500,4.760340,2.048761,34.882900,12.706400,27.696300,27.705500
2250,4.738583,2.039718,35.520700,13.428300,28.275500,28.301000
3000,4.710003,2.038700,35.630400,13.584800,28.385500,28.414000



Step 500/3125 (16.0%)
loss: 5.2268
grad_norm: 1.4310
learning_rate: 0.0002
epoch: 0.1600
Elapsed: 9.5 min
GPU memory: 0.55/14.56 GB used | Reserved: 9.61 GB | Peak: 5.77 GB

Step 750/3125 (24.0%)
eval_loss: 2.0726
eval_rouge1: 34.1768
eval_rouge2: 12.3456
eval_rougeL: 27.2087
eval_rougeLsum: 27.2455
eval_runtime: 138.4571
eval_samples_per_second: 7.2220
eval_steps_per_second: 0.4550
epoch: 0.2400
Elapsed: 16.6 min
GPU memory: 0.55/14.56 GB used | Reserved: 2.92 GB | Peak: 5.59 GB

Step 1000/3125 (32.0%)
loss: 4.8094
grad_norm: 1.5324
learning_rate: 0.0001
epoch: 0.3200
Elapsed: 21.5 min
GPU memory: 0.55/14.56 GB used | Reserved: 7.86 GB | Peak: 5.96 GB

Step 1500/3125 (48.0%)
loss: 4.7603
grad_norm: 1.5628
learning_rate: 0.0001
epoch: 0.4800
Elapsed: 31.2 min
GPU memory: 0.55/14.56 GB used | Reserved: 7.87 GB | Peak: 5.76 GB

Step 1500/3125 (48.0%)
eval_loss: 2.0488
eval_rouge1: 34.8829
eval_rouge2: 12.7064
eval_rougeL: 27.6963
eval_rougeLsum: 27.7055
eval_runtime: 136.1080
eval_sampl

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
750,5.226796,2.072565,34.176800,12.345600,27.208700,27.245500
1500,4.760340,2.048761,34.882900,12.706400,27.696300,27.705500
2250,4.738583,2.039718,35.520700,13.428300,28.275500,28.301000
3000,4.710003,2.038700,35.630400,13.584800,28.385500,28.414000
3125,4.710003,2.037392,35.623600,13.537700,28.317400,28.336900



Step 3125/3125 (100.0%)
eval_loss: 2.0374
eval_rouge1: 35.6236
eval_rouge2: 13.5377
eval_rougeL: 28.3174
eval_rougeLsum: 28.3369
eval_runtime: 136.0730
eval_samples_per_second: 7.3490
eval_steps_per_second: 0.4630
epoch: 1.0000
Elapsed: 71.7 min
GPU memory: 0.55/14.56 GB used | Reserved: 2.92 GB | Peak: 5.76 GB

Step 3125/3125 (100.0%)
train_runtime: 4303.2577
train_samples_per_second: 23.2380
train_steps_per_second: 0.7260
total_flos: 59347998817910784.0000
train_loss: 4.8204
epoch: 1.0000
Elapsed: 71.7 min
GPU memory: 0.54/14.56 GB used | Reserved: 2.92 GB | Peak: 0.55 GB

Training finished | Total time: 71.7 min
GPU memory: 0.54/14.56 GB used | Reserved: 2.92 GB | Peak: 0.55 GB

Training metrics:
train_runtime: 4303.2577
train_samples_per_second: 23.238
train_steps_per_second: 0.726
total_flos: 5.934799881791078e+16
train_loss: 4.820442421875
epoch: 1.0
***** train metrics *****
  epoch                    =        1.0
  total_flos               = 55272131GF
  train_loss            

In [ ]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

trainer_output/checkpoint-3000
28.414


# **8- Test set evaluation**

In [ ]:
#Tokenize test set
tokenized_test = dataset["test"].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["test"].column_names,
    desc="Tokenizing test",
)

#Evaluate on up to 1000 examples
n = min(1000, len(tokenized_test))
test_subset = tokenized_test.shuffle(seed=SEED).select(range(n))

results = trainer.evaluate(
    eval_dataset=test_subset,
    metric_key_prefix="test",
)

print(results)

Tokenizing test:   0%|          | 0/11334 [00:00<?, ? examples/s]


Step 3125/3125 (100.0%)
test_loss: 2.0350
test_rouge1: 35.6910
test_rouge2: 13.2733
test_rougeL: 28.3700
test_rougeLsum: 28.4071
test_runtime: 128.9038
test_samples_per_second: 7.7580
test_steps_per_second: 0.4890
epoch: 1.0000
Elapsed: 75.2 min
GPU memory: 0.54/14.56 GB used | Reserved: 2.53 GB | Peak: 1.95 GB


Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel,Rougelsum
4.710003,2.035004,3125,35.691000,13.273300,28.370000,28.407100


{'test_loss': 2.035004138946533, 'test_rouge1': 35.691, 'test_rouge2': 13.2733, 'test_rougeL': 28.37, 'test_rougeLsum': 28.4071}


# **9- Qualitative examples**

In [ ]:
def show_predictions(n=5, max_input_len=1024, max_new_tokens=128, num_beams=2, seed=40):
    """Generate and display n random test predictions vs references."""
    model.eval()
    device = next(model.parameters()).device

    test = dataset["test"]
    rng = random.Random(seed)
    idxs = rng.sample(range(len(test)), n)

    for i, idx in enumerate(idxs, 1):
        ex = test[idx]
        article = ex["document"]
        reference = ex["summary"]

        #Encode a single article into a batched tensor on-device
        enc = tokenizer(
            article,
            max_length=max_input_len,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                early_stopping=True,
            )

        pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()

        print("=" * 100)
        print(f"[{i}] Article:\n{article[:500]}...")
        print(f"\nREFERENCE: {reference}")
        print(f"PREDICTED: {pred}")

show_predictions(n=5)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] Article:
At least 100 others were wounded in the strike, which reportedly struck a warehouse containing weapons.
The commando unit is allied with Houthi rebels whose military capabilities the coalition has been trying to destroy.
Casualties were also reported after air strikes in the northern province of Hajjah, near the Saudi border.
Residents of the Bakeel al-Meer area told the Reuters news agency that at least 40 people had been killed, mostly civilians.
"Houthi gunmen were attacking Saudi border posi...

REFERENCE: Saudi-led coalition warplanes have bombed a special forces camp in Yemen's capital, Sanaa, killing at least 36 people, officials and witnesses say.
PREDICTED: At least 40 people have been killed in air strikes by the Saudi-led coalition against Houthi rebels in Yemen, officials say.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2] Article:
Scientists trying to understand the rise of antibiotic resistance, considered by some to be the great health threat of our time, believe they could provide clues to the mechanisms behind its spread.
In an increasingly global society, travel can be pointed to as one of the reasons for the rise of antibiotic resistance.
Now an international team of researchers from Newcastle University and the Indian Institute of Technology, Delhi, is studying how large gatherings of people could become hotspots f...

REFERENCE: The Hindu holy sites in the Indian cities of Rishikesh and Haridwar attract millions of pilgrims each year - but they are now the destination for another group of devotees.
PREDICTED: A new study has found that large gatherings of people can become hotspots for the spread of antibiotic resistance.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[3] Article:
The A547 at Meliden, between Rhuddlan and Prestatyn, was closed after 14:00 BST on Saturday after the collision.
The child was taken to hospital, with the extent of his injuries not yet known.
Police confirmed traffic had resumed on the road after 16:00 BST....

REFERENCE: A Denbighshire road has been reopened after a boy was hit by a car, North Wales Police says.
PREDICTED: A child has been taken to hospital after being hit by a car on the A547.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[4] Article:
New analysis for BBC News shows that many people relying on their savings income are worse off than ever before.
Savings rates plummeted after the Bank of England slashed its base rate in the financial crisis.
Since last autumn, as the economic outlook has worsened, they have fallen again.
Tax-free Isas, fixed rate bonds and easy access accounts are all at or near their lowest points.
In research carried out for the BBC, the rate-checking firm Savings Champion recorded 1,440 savings rate cuts la...

REFERENCE: Interest rates for savers have fallen to new record lows, after hundreds of cuts in recent months and more than 1,000 in the past year.
PREDICTED: Savings rates are at their lowest levels since the financial crisis, according to a new study.
[5] Article:
Lindsay Rimer, 13, was found in the Rochdale Canal five months after she disappeared in Hebden Bridge in November 1994.
Despite extensive police inquiries, her killer has never been found.
Kate Rimer said not knowing